In [16]:
# ============================================================
# INTEGRATED CRACK ANALYSIS SYSTEM (YOLO + DATASET + HYBRID)
# ============================================================

from ultralytics import YOLO
import cv2
import numpy as np
import gradio as gr
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
MODEL_PATH = "/Users/Apple/Desktop/project/runs/classify/crack_classifier_fast/weights/best.pt"
DATASET_PATH = "/Users/Apple/Downloads/carck_width_datset (2).xlsx"

PIXELS_PER_MM = 10.0
MIN_CONTOUR_AREA = 50

# ============================================================
# 1. LOAD YOLO CLASSIFIER
# ============================================================
yolo_model = YOLO(MODEL_PATH)

def yolo_classify(image):
    results = yolo_model.predict(image, imgsz=224, verbose=False)[0]
    probs = results.probs
    class_id = int(probs.top1)
    label = yolo_model.names[class_id]
    conf = float(probs.top1conf)

    label_clean = label.replace("_", " ").upper()
    is_crack = "crack" in label.lower() and "no" not in label.lower()

    return is_crack, label_clean, conf


# ============================================================
# 2. LOAD DATASET + TRAIN ML MODELS
# ============================================================

# Column mapping to clean names
COL_MAP = {
    "FCK_MPa":        "FCK(mpa)",
    "FS_MPa":         "FS",
    "NUM_BARS":       "NO OF BARS",
    "B_mm":           "B(mm)",
    "D_mm":           "D(mm)",
    "BAR_SPACE":      "bar space",
    "DIA_mm":         "DIA(MM)",
    "RATIO_val":      "RATIO",
    "COVER_mm":       "COVER DEPTH(MM)",
    "CRACK_WIDTH_mm": "MAX WIDTH(mm)",
    "SEVERITY":       "type",
}

df_raw = pd.read_excel(DATASET_PATH)
df_raw.columns = df_raw.columns.str.strip()

rename_dict = {old: new for new, old in COL_MAP.items() if old in df_raw.columns}
data = df_raw.rename(columns=rename_dict)

# ------------------------------
# Crack Width Regression Model
# ------------------------------
FEATURE_COLS_WIDTH = ["FCK_MPa", "FS_MPa", "NUM_BARS", "D_mm", "COVER_mm", "DIA_mm"]
TARGET_COL_WIDTH = "CRACK_WIDTH_mm"

data_width = data.dropna(subset=FEATURE_COLS_WIDTH + [TARGET_COL_WIDTH])
X_w = data_width[FEATURE_COLS_WIDTH].values
y_w = data_width[TARGET_COL_WIDTH].values

width_model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
width_model.fit(X_w, y_w)

# ------------------------------
# Severity Classifier Model
# ------------------------------
FEATURE_COLS_SEV = FEATURE_COLS_WIDTH + [TARGET_COL_WIDTH]
TARGET_COL_SEV = "SEVERITY"

data_sev = data.dropna(subset=FEATURE_COLS_SEV + [TARGET_COL_SEV])
X_s = data_sev[FEATURE_COLS_SEV].values
y_s_raw = data_sev[TARGET_COL_SEV].astype(str).values

label_encoder = LabelEncoder()
y_s = label_encoder.fit_transform(y_s_raw)

severity_model = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
severity_model.fit(X_s, y_s)


# ML Predict Functions
def ml_predict_width(FCK, D, COVER, FS, NO_OF_BARS, DIA):
    x = np.array([[FCK, FS, NO_OF_BARS, D, COVER, DIA]], dtype=float)
    return float(width_model.predict(x)[0])


def ml_predict_severity(FCK, D, COVER, FS, NO_OF_BARS, DIA, width_used):
    x = np.array([[FCK, FS, NO_OF_BARS, D, COVER, DIA, width_used]], dtype=float)
    code = severity_model.predict(x)[0]
    return label_encoder.inverse_transform([code])[0]


# ============================================================
# 3. SEVERITY MAPPING (Dataset labels → Human labels)
# ============================================================

SEVERITY_MAP = {
    "F": "Failure",
    "T": "Tolerable"
}

def map_severity(label):
    return SEVERITY_MAP.get(str(label).strip(), label)


# ============================================================
# 4. HYBRID ENGINEERING + ML SEVERITY LOGIC
# ============================================================

def hybrid_severity(FCK, D, COVER, FS, NO_OF_BARS, DIA, measured_width, measured_length):

    width_ml = ml_predict_width(FCK, D, COVER, FS, NO_OF_BARS, DIA)
    width_used = max(measured_width, width_ml)

    ml_raw = ml_predict_severity(FCK, D, COVER, FS, NO_OF_BARS, DIA, width_used)
    ml_label = map_severity(ml_raw)

    # Engineering thresholds
    TOL_LIMIT = 0.30     # mm (IS 456)
    FAIL_LIMIT = 0.40    # mm (ACI)

    # RULE 1 — Small cracks ALWAYS tolerable
    if width_used <= TOL_LIMIT:
        return "Tolerable"

    # RULE 2 — Large cracks ALWAYS failure
    if width_used >= FAIL_LIMIT:
        return "Failure"

    # RULE 3 — Mid cracks → let ML decide
    return ml_label


# ============================================================
# 5. CRACK FEATURE EXTRACTION
# ============================================================

def extract_crack_features(image):
    if image is None:
        return None, "No image uploaded.", 0.0, 0.0

    rgb = image.copy().astype(np.uint8)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    thresh = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 21, 7
    )

    kernel = np.ones((3, 3), np.uint8)
    clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = [c for c in contours if cv2.contourArea(c) >= MIN_CONTOUR_AREA]

    annotated = rgb.copy()

    if not contours:
        return annotated, "No crack-like region detected.", 0.0, 0.0

    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    table = ["ID    Length(mm)    Width(mm)"]
    max_w_mm = 0
    max_L_mm = 0

    for idx, cnt in enumerate(contours, start=1):
        x, y, w, h = cv2.boundingRect(cnt)

        length_mm = max(w, h) / PIXELS_PER_MM
        width_mm = min(w, h) / PIXELS_PER_MM

        max_w_mm = max(max_w_mm, width_mm)
        max_L_mm = max(max_L_mm, length_mm)

        cv2.drawContours(annotated, [cnt], -1, (255, 20, 20), 1)

        cx = x + w // 2
        cy = y + h // 2
        cv2.putText(annotated, f"C{idx}", (cx-12, cy+5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        table.append(f"C{idx:<3}   {length_mm:>8.2f}      {width_mm:>8.2f}")

    info_text = "\n".join(table)
    return annotated, info_text, max_w_mm, max_L_mm


# ============================================================
# 6. FULL PIPELINE
# ============================================================

def full_pipeline(image, FCK, D, COVER, FS, NO_OF_BARS, DIA):

    if image is None:
        return None, "No image", "No data", "No crack"

    is_crack, label_clean, conf = yolo_classify(image)
    pred_text = f"{label_clean} | CONF: {conf*100:.1f}%"

    if not is_crack:
        return image, pred_text, "No cracks detected", "Overall Crack Severity: NO CRACK"

    annotated, crack_info, max_w_mm, max_L_mm = extract_crack_features(image)

    severity_label = hybrid_severity(
        FCK, D, COVER, FS, NO_OF_BARS, DIA,
        measured_width=max_w_mm,
        measured_length=max_L_mm
    )

    final_severity = f"Overall Crack Severity: {severity_label}"

    return annotated, pred_text, crack_info, final_severity


# ============================================================
# 7. GRADIO UI
# ============================================================

with gr.Blocks() as demo:
    gr.Markdown("# 🧱 Integrated Crack Analysis System (YOLO + Dataset + Hybrid Severity)")

    with gr.Row():
        with gr.Column(scale=3):
            img_in = gr.Image(label="Upload Image", type="numpy")
            FCK_in = gr.Number(label="FCK (MPa)", value=25)
            D_in = gr.Number(label="D (mm)", value=300)
            COVER_in = gr.Number(label="Cover (mm)", value=40)
            FS_in = gr.Number(label="FS (MPa)", value=240)
            NBAR_in = gr.Number(label="Bars", value=3)
            DIA_in = gr.Number(label="Diameter (mm)", value=16)
            btn = gr.Button("Analyze", variant="primary")

        with gr.Column(scale=4):
            img_out = gr.Image(label="Result Image", type="numpy", width=900)

    with gr.Row():
        txt_pred = gr.Textbox(label="YOLO Prediction")
        txt_dims = gr.Textbox(label="Crack Measurements (mm)", lines=8)
        txt_sev = gr.Textbox(label="Severity")

    btn.click(
        fn=full_pipeline,
        inputs=[img_in, FCK_in, D_in, COVER_in, FS_in, NBAR_in, DIA_in],
        outputs=[img_out, txt_pred, txt_dims, txt_sev]
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.
